# Phase 22: Feature Selection & Validation

**Goal:** After Phase 18, 19, and 20, we have engineered dozens of mathematically advanced features. If we add those to the 78 raw features in CICIDS-2017, we have over 100 features!

Giving an AI too many features causes **Overfitting** (the AI memorizes noise instead of learning the pattern) and ruins **Explainability** (SHAP gets confused by redundant data). We must rigorously filter out bad features using a Three-Stage Mathematical Pipeline!

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd  # type: ignore  # pylint: disable=import-error
import numpy as np  # type: ignore  # pylint: disable=import-error
from sklearn.feature_selection import VarianceThreshold, SelectKBest, mutual_info_classif  # type: ignore  # pylint: disable=import-error
import yaml  # type: ignore  # pylint: disable=import-error
import os  # type: ignore  # pylint: disable=import-error

pd.set_option('display.max_columns', None)

### Stage 1: Variance Threshold
If a feature has a variance of `< 0.01`, it means almost every single row has the exact same number. If a feature never changes, the AI cannot learn anything from it. We drop it immediately.

In [2]:
def filter_by_variance(df: pd.DataFrame, threshold: float = 0.01):
    selector = VarianceThreshold(threshold=threshold)
    selector.fit(df)
    
    # Get the names of the features we are KEEPING
    kept_features = df.columns[selector.get_support()]
    removed_features = [col for col in df.columns if col not in kept_features]
    
    return df[kept_features], removed_features

### Stage 2: Mutual Information (MI) Ranking
Mutual Information measures how much a feature can "predict" the Target Label (`is_attack`). If a feature is completely random noise, its MI score is 0. If it perfectly predicts the attack, its MI score is high.
We calculate the MI score for everything and drop the useless ones.

In [3]:
def calculate_mutual_information(X: pd.DataFrame, y: pd.Series):
    mi_scores = mutual_info_classif(X, y, random_state=42)
    mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
    return mi_series

def filter_by_mutual_information(X: pd.DataFrame, mi_series: pd.Series, top_k: int = 50):
    kept_features = mi_series.head(top_k).index.tolist()
    removed_features = [col for col in X.columns if col not in kept_features]
    return X[kept_features], removed_features

### Stage 3: Correlation Redundancy Filter
If Feature A and Feature B are 99% identical (Pearson correlation > 0.95), we don't need both! 
We look at the correlated pair, check which one has the higher **Mutual Information (MI) Score**, keep the winner, and drop the loser!

In [4]:
def filter_by_correlation(df: pd.DataFrame, mi_series: pd.Series, threshold: float = 0.95):
    corr_matrix = df.corr().abs()
    
    # Get the upper triangle of the correlation matrix
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    removed_features = []
    for col in upper.columns:
        highly_correlated_with = upper.index[upper[col] > threshold].tolist()
        
        for correlated_col in highly_correlated_with:
            # If they are correlated, check who has the higher MI score
            mi_col1 = mi_series[col]
            mi_col2 = mi_series[correlated_col]
            
            # Drop the loser
            if mi_col1 < mi_col2 and col not in removed_features:
                removed_features.append(col)
            elif correlated_col not in removed_features:
                removed_features.append(correlated_col)
                
    kept_features = [c for c in df.columns if c not in removed_features]
    return df[kept_features], removed_features

### Step 4: Three-Stage Pipeline Integration Test (Subphase 22.1)
Let's create a simulated dataset to prove this pipeline works flawlessly!

In [5]:
np.random.seed(42)

fake_data = pd.DataFrame({
    "constant_feature": [0, 0, 0, 0, 0, 0],                            # Should fail Stage 1 (Variance)
    "useful_feature":   [1, 2, 3, 10, 11, 12],                         # Highly predictive of the attack
    "redundant_feat":   [1.1, 2.1, 3.1, 10.1, 11.1, 12.1],             # 99% Correlated with useful_feature! Should fail Stage 3.
    "random_noise":     [99, 3, 42, 7, 88, 14],                        # Zero correlation to the attack. Should fail Stage 2 (MI).
    "attack_label":     [0, 0, 0, 1, 1, 1]                             # 0 = Normal, 1 = Hacker
})

X = fake_data.drop(columns=["attack_label"])
y = fake_data["attack_label"]

print("=== STARTING 3-STAGE FEATURE SELECTION ===")
print(f"Initial Features: {X.columns.tolist()}\n")

# Stage 1
X_var, dropped_var = filter_by_variance(X)
print(f"[Stage 1] Dropped by Variance: {dropped_var}")

# Stage 2
mi_scores = calculate_mutual_information(X_var, y)
X_mi, dropped_mi = filter_by_mutual_information(X_var, mi_scores, top_k=2) # Keep top 2 for this test
print(f"[Stage 2] Dropped by Mutual Info: {dropped_mi}")

# Stage 3
X_final, dropped_corr = filter_by_correlation(X_mi, mi_scores)
print(f"[Stage 3] Dropped by Correlation: {dropped_corr}\n")

print("✅ PIPELINE SUCCESS!")
print(f"Final Selected Features: {X_final.columns.tolist()}")

=== STARTING 3-STAGE FEATURE SELECTION ===
Initial Features: ['constant_feature', 'useful_feature', 'redundant_feat', 'random_noise']

[Stage 1] Dropped by Variance: ['constant_feature']
[Stage 2] Dropped by Mutual Info: ['random_noise']
[Stage 3] Dropped by Correlation: ['useful_feature']

✅ PIPELINE SUCCESS!
Final Selected Features: ['redundant_feat']


### Step 5: Documenting the Selected Features (Subphase 22.2)
In a research environment, the final list of selected features is a critical artifact. All six of our AI models must load this exact same YAML file to guarantee they are training on identical feature columns!

In [6]:
# Generate the exact yaml structure required by the subphase
yaml_data = {
    "feature_names": X_final.columns.tolist(),
    "feature_count": len(X_final.columns),
    "selection_date": "2024-03-15",
    "dataset_version": "v1.0",
    "stages_applied": ["VarianceThreshold", "MutualInformation", "CorrelationRedundancy"],
    "removed_by_variance": dropped_var,
    "removed_by_mi": dropped_mi,
    "removed_by_correlation": dropped_corr
}

os.makedirs("../configs", exist_ok=True)
with open("../configs/selected_features.yaml", "w") as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print("✅ SUCCESS: Saved to ml/configs/selected_features.yaml!")
print("Every model training script will now dynamically load its feature list from this file.")

✅ SUCCESS: Saved to ml/configs/selected_features.yaml!
Every model training script will now dynamically load its feature list from this file.
